In [14]:

!pip install -q ipywidgets matplotlib pandas

from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import display, clear_output, HTML
from collections import deque

print("OS Scheduling Visualizer setup completed!")


OS Scheduling Visualizer setup completed!


In [15]:

# ============================================================
# OS SCHEDULING VISUALIZER
# DISK SCHEDULING AND CPU SCHEDULING
# ============================================================

# ============================================================
# DISK SCHEDULING INPUTS
# ============================================================

disk_algorithm = widgets.Dropdown(
    options=["FCFS", "SSTF", "SCAN", "C-SCAN", "LOOK"],
    value="FCFS",
    description="Algorithm:"
)

disk_queue = widgets.Text(
    value="82,170,43,140,24,16,190",
    description="Request Queue:",
    layout=widgets.Layout(width="500px")
)

disk_head = widgets.IntText(
    value=50,
    description="Initial Head:"
)

disk_size = widgets.IntText(
    value=200,
    description="Cylinders:"
)

disk_direction = widgets.Dropdown(
    options=["Right", "Left"],
    value="Right",
    description="Direction:"
)

disk_button = widgets.Button(
    description="Run Disk Scheduling",
    button_style="success",
    icon="play"
)

disk_output = widgets.Output()


# ============================================================
# DISK STEP CONTROLS
# ============================================================

disk_play = widgets.Play(
    value=0,
    min=0,
    max=0,
    step=1,
    interval=1200
)

disk_previous = widgets.Button(
    description="Previous",
    icon="step-backward"
)

disk_next = widgets.Button(
    description="Next",
    icon="step-forward"
)

disk_load = widgets.Button(
    description="Load Disk Steps",
    button_style="info"
)

disk_step_output = widgets.Output()

disk_state = {
    "loaded": False,
    "head": 0,
    "sequence": [],
    "algorithm": ""
}


# ============================================================
# DISK SCHEDULING ALGORITHM
# ============================================================

def disk_scheduling(head, requests, size, algorithm, direction):

    current = head
    sequence = []

    left = sorted([x for x in requests if x < head])
    right = sorted([x for x in requests if x >= head])

    if algorithm == "FCFS":

        sequence = requests.copy()

    elif algorithm == "SSTF":

        pending = requests.copy()

        while pending:

            nearest = min(
                pending,
                key=lambda x: abs(x - current)
            )

            sequence.append(nearest)
            pending.remove(nearest)
            current = nearest

    elif algorithm == "SCAN":

        if direction == "Right":

            sequence = right.copy()

            if not sequence or sequence[-1] != size - 1:
                sequence.append(size - 1)

            sequence += left[::-1]

        else:

            sequence = left[::-1]

            if not sequence or sequence[-1] != 0:
                sequence.append(0)

            sequence += right

    elif algorithm == "C-SCAN":

        if direction == "Right":

            sequence = right.copy()

            if left:

                if not sequence or sequence[-1] != size - 1:
                    sequence.append(size - 1)

                sequence.append(0)
                sequence += left

        else:

            sequence = left[::-1]

            if right:

                if not sequence or sequence[-1] != 0:
                    sequence.append(0)

                sequence.append(size - 1)
                sequence += right[::-1]

    elif algorithm == "LOOK":

        if direction == "Right":
            sequence = right + left[::-1]

        else:
            sequence = left[::-1] + right

    movement = 0
    current = head

    movements = []

    for cylinder in sequence:

        distance = abs(current - cylinder)

        movement += distance

        movements.append({
            "From": current,
            "To": cylinder,
            "Movement": distance
        })

        current = cylinder

    return sequence, movement, movements


# ============================================================
# DISK RUN BUTTON
# ============================================================

def run_disk(button):

    with disk_output:

        clear_output(wait=True)

        try:

            requests = [
                int(x.strip())
                for x in disk_queue.value.split(",")
                if x.strip()
            ]

            head = disk_head.value
            size = disk_size.value
            algorithm = disk_algorithm.value
            direction = disk_direction.value

            if not requests:
                raise ValueError("Enter disk requests.")

            if size <= 0:
                raise ValueError("Cylinder size must be positive.")

            if not 0 <= head < size:
                raise ValueError("Invalid initial head position.")

            if any(x < 0 or x >= size for x in requests):
                raise ValueError("Requests must be within the cylinder range.")

            sequence, total, movements = disk_scheduling(
                head, requests, size, algorithm, direction
            )

            print("DISK SCHEDULING RESULTS")
            print("=" * 40)

            print("Algorithm:", algorithm)
            print("Initial Head:", head)
            print("Request Queue:", requests)
            print("Head Movement Order:", [head] + sequence)

            print("\nMovement Details:")
            display(pd.DataFrame(movements))

            print("\nTotal Head Movement:", total, "cylinders")

            positions = [head] + sequence

            plt.figure(figsize=(10, 4))

            plt.plot(
                range(len(positions)),
                positions,
                marker="o",
                linewidth=2
            )

            for i, value in enumerate(positions):

                plt.annotate(
                    str(value),
                    (i, value),
                    textcoords="offset points",
                    xytext=(0, 8),
                    ha="center"
                )

            plt.title(algorithm + " Disk Scheduling")
            plt.xlabel("Movement Step")
            plt.ylabel("Cylinder Number")
            plt.grid(True, alpha=0.3)

            plt.show()
            plt.close()

            if requests == [82, 170, 43, 140, 24, 16, 190] and head == 50:

                print("\nSample Verification:")
                print("Expected FCFS = 642 cylinders")
                print("Expected SSTF = 208 cylinders")

                if algorithm == "FCFS":
                    print("Calculated FCFS:", total)

                elif algorithm == "SSTF":
                    print("Calculated SSTF:", total)

        except Exception as error:

            print("Input Error:", error)


disk_button.on_click(run_disk)


# ============================================================
# DISK STEP-BY-STEP DISPLAY
# ============================================================

def render_disk_step(change=None):

    with disk_step_output:

        clear_output(wait=True)

        if not disk_state["loaded"]:

            print("Click Load Disk Steps to begin.")
            return

        step = disk_play.value

        head = disk_state["head"]
        sequence = disk_state["sequence"]

        positions = [head] + sequence[:step]

        total_movement = sum(
            abs(positions[i] - positions[i-1])
            for i in range(1, len(positions))
        )

        print("DISK SCHEDULING:", disk_state["algorithm"])
        print("=" * 40)

        print("Current Step:", step, "of", len(sequence))
        print("Current Head Position:", positions[-1])

        print("Movement Order:", positions)

        print("Total Movement So Far:", total_movement, "cylinders")

        if step == 0:

            print("\nStarting position.")

        else:

            print(
                "\nPrevious Position:",
                positions[-2]
            )

            print(
                "Current Position:",
                positions[-1]
            )

            print(
                "Movement:",
                abs(positions[-1] - positions[-2]),
                "cylinders"
            )

        plt.figure(figsize=(9, 3.5))

        plt.plot(
            range(len(positions)),
            positions,
            marker="o",
            linewidth=2
        )

        for i, value in enumerate(positions):

            plt.annotate(
                str(value),
                (i, value),
                textcoords="offset points",
                xytext=(0, 8),
                ha="center"
            )

        plt.title("Disk Head Movement - Step by Step")
        plt.xlabel("Movement Step")
        plt.ylabel("Cylinder Number")
        plt.grid(True, alpha=0.3)

        plt.show()
        plt.close()


def load_disk_steps(button):

    try:

        requests = [
            int(x.strip())
            for x in disk_queue.value.split(",")
            if x.strip()
        ]

        head = disk_head.value
        size = disk_size.value

        if not requests:
            raise ValueError("Enter disk requests.")

        if size <= 0 or not 0 <= head < size:
            raise ValueError("Invalid disk input.")

        if any(x < 0 or x >= size for x in requests):
            raise ValueError("Invalid cylinder request.")

        sequence, total, movements = disk_scheduling(
            head,
            requests,
            size,
            disk_algorithm.value,
            disk_direction.value
        )

        disk_state["loaded"] = True
        disk_state["head"] = head
        disk_state["sequence"] = sequence
        disk_state["algorithm"] = disk_algorithm.value

        disk_play.max = len(sequence)
        disk_play.value = 0

        render_disk_step()

    except Exception as error:

        with disk_step_output:
            clear_output(wait=True)
            print("Input Error:", error)


def disk_prev_click(button):

    disk_play.value = max(0, disk_play.value - 1)


def disk_next_click(button):

    disk_play.value = min(
        disk_play.max,
        disk_play.value + 1
    )


disk_load.on_click(load_disk_steps)
disk_previous.on_click(disk_prev_click)
disk_next.on_click(disk_next_click)

disk_play.observe(
    render_disk_step,
    names="value"
)


# ============================================================
# CPU SCHEDULING INPUTS
# ============================================================

cpu_algorithm = widgets.Dropdown(
    options=["FCFS", "SJF", "SRTF", "Priority", "Round Robin"],
    value="FCFS",
    description="Algorithm:"
)

cpu_burst = widgets.Text(
    value="5,3,4,2",
    description="Burst Times:",
    layout=widgets.Layout(width="400px")
)

cpu_arrival = widgets.Text(
    value="0,1,2,3",
    description="Arrival Times:",
    layout=widgets.Layout(width="400px")
)

cpu_priority = widgets.Text(
    value="2,1,3,2",
    description="Priority:",
    layout=widgets.Layout(width="400px")
)

cpu_quantum = widgets.IntSlider(
    value=2,
    min=1,
    max=10,
    description="Time Quantum:"
)

cpu_button = widgets.Button(
    description="Run CPU Scheduling",
    button_style="success",
    icon="play"
)

cpu_output = widgets.Output()


# ============================================================
# CPU STEP CONTROLS
# ============================================================

cpu_play = widgets.Play(
    value=0,
    min=0,
    max=0,
    step=1,
    interval=1000
)

cpu_previous = widgets.Button(
    description="Previous",
    icon="step-backward"
)

cpu_next = widgets.Button(
    description="Next",
    icon="step-forward"
)

cpu_load = widgets.Button(
    description="Load CPU Steps",
    button_style="info"
)

cpu_step_output = widgets.Output()

cpu_state = {
    "loaded": False,
    "segments": [],
    "algorithm": "",
    "total": 0
}


# ============================================================
# ADD CPU GANTT CHART SEGMENT
# ============================================================

def add_segment(segments, process, start, end):

    if end <= start:
        return

    if (
        segments
        and segments[-1]["process"] == process
        and segments[-1]["end"] == start
    ):

        segments[-1]["end"] = end

    else:

        segments.append({
            "process": process,
            "start": start,
            "end": end
        })


# ============================================================
# CPU SCHEDULING ALGORITHM
# ============================================================

def cpu_scheduling(bursts, arrivals, priorities, algorithm, quantum):

    n = len(bursts)

    remaining = bursts.copy()

    first_start = [None] * n
    completion = [0] * n

    segments = []

    completed = 0
    time = 0

    # FCFS, SJF AND PRIORITY

    if algorithm in ["FCFS", "SJF", "Priority"]:

        done = [False] * n

        while completed < n:

            available = [
                i for i in range(n)
                if arrivals[i] <= time and not done[i]
            ]

            if not available:

                future = [
                    arrivals[i] for i in range(n)
                    if not done[i]
                ]

                next_time = min(future)

                add_segment(
                    segments, "IDLE", time, next_time
                )

                time = next_time
                continue

            if algorithm == "FCFS":

                p = min(
                    available,
                    key=lambda i: (arrivals[i], i)
                )

            elif algorithm == "SJF":

                p = min(
                    available,
                    key=lambda i: (bursts[i], arrivals[i], i)
                )

            else:

                p = min(
                    available,
                    key=lambda i: (priorities[i], arrivals[i], i)
                )

            first_start[p] = time

            start = time
            time += remaining[p]

            add_segment(
                segments, f"P{p+1}", start, time
            )

            remaining[p] = 0
            completion[p] = time

            done[p] = True
            completed += 1

    # SRTF

    elif algorithm == "SRTF":

        while completed < n:

            available = [
                i for i in range(n)
                if arrivals[i] <= time and remaining[i] > 0
            ]

            if not available:

                future = [
                    arrivals[i] for i in range(n)
                    if remaining[i] > 0
                ]

                next_time = min(future)

                add_segment(
                    segments, "IDLE", time, next_time
                )

                time = next_time
                continue

            p = min(
                available,
                key=lambda i: (remaining[i], arrivals[i], i)
            )

            if first_start[p] is None:
                first_start[p] = time

            add_segment(
                segments, f"P{p+1}", time, time + 1
            )

            remaining[p] -= 1
            time += 1

            if remaining[p] == 0:

                completion[p] = time
                completed += 1

    # ROUND ROBIN

    elif algorithm == "Round Robin":

        ready = deque()
        added = [False] * n

        while completed < n:

            for i in range(n):

                if arrivals[i] <= time and not added[i]:

                    ready.append(i)
                    added[i] = True

            if not ready:

                future = [
                    arrivals[i] for i in range(n)
                    if not added[i]
                ]

                next_time = min(future)

                add_segment(
                    segments, "IDLE", time, next_time
                )

                time = next_time
                continue

            p = ready.popleft()

            if first_start[p] is None:
                first_start[p] = time

            run_time = min(quantum, remaining[p])

            start = time
            time += run_time

            remaining[p] -= run_time

            add_segment(
                segments, f"P{p+1}", start, time
            )

            for i in range(n):

                if arrivals[i] <= time and not added[i]:

                    ready.append(i)
                    added[i] = True

            if remaining[p] > 0:

                ready.append(p)

            else:

                completion[p] = time
                completed += 1

    # CALCULATE RESULTS

    results = []

    for i in range(n):

        tat = completion[i] - arrivals[i]
        wt = tat - bursts[i]
        rt = first_start[i] - arrivals[i]

        results.append({

            "Process": f"P{i+1}",
            "AT": arrivals[i],
            "BT": bursts[i],
            "Priority": priorities[i],
            "CT": completion[i],
            "TAT": tat,
            "WT": wt,
            "RT": rt

        })

    return segments, results


# ============================================================
# DRAW COMPLETE CPU GANTT CHART
# ============================================================

def draw_cpu_chart(segments):

    fig, ax = plt.subplots(figsize=(12, 3.5))

    for seg in segments:

        start = seg["start"]
        end = seg["end"]
        name = seg["process"]

        ax.barh(
            0,
            end - start,
            left=start,
            height=0.5,
            edgecolor="black",
            alpha=0.85
        )

        ax.text(
            (start + end) / 2,
            0,
            name,
            ha="center",
            va="center",
            fontsize=9
        )

        ax.text(
            start,
            -0.38,
            str(start),
            ha="center",
            fontsize=9
        )

    if segments:

        ax.text(
            segments[-1]["end"],
            -0.38,
            str(segments[-1]["end"]),
            ha="center",
            fontsize=9
        )

    ax.set_yticks([])
    ax.set_xlabel("CPU Time (ms)")
    ax.set_title("CPU Scheduling Gantt Chart")
    ax.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close(fig)


# ============================================================
# CPU RUN BUTTON
# ============================================================

def run_cpu(button):

    with cpu_output:

        clear_output(wait=True)

        try:

            bursts = [
                int(x.strip())
                for x in cpu_burst.value.split(",")
                if x.strip()
            ]

            arrivals = [
                int(x.strip())
                for x in cpu_arrival.value.split(",")
                if x.strip()
            ]

            priorities = [
                int(x.strip())
                for x in cpu_priority.value.split(",")
                if x.strip()
            ]

            algorithm = cpu_algorithm.value
            quantum = cpu_quantum.value

            n = len(bursts)

            if n == 0 or any(x <= 0 for x in bursts):
                raise ValueError("Burst times must be positive.")

            if len(arrivals) != n:
                raise ValueError("Enter one arrival time per process.")

            if len(priorities) != n:
                raise ValueError("Enter one priority per process.")

            if any(x < 0 for x in arrivals):
                raise ValueError("Arrival times cannot be negative.")

            segments, results = cpu_scheduling(
                bursts,
                arrivals,
                priorities,
                algorithm,
                quantum
            )

            df = pd.DataFrame(results)

            print("CPU SCHEDULING RESULTS")
            print("=" * 40)

            print("Selected Algorithm:", algorithm)

            print("\nProcess Table:")
            display(df)

            print(
                "\nAverage Waiting Time:",
                round(df["WT"].mean(), 2), "ms"
            )

            print(
                "Average Turnaround Time:",
                round(df["TAT"].mean(), 2), "ms"
            )

            print(
                "Average Response Time:",
                round(df["RT"].mean(), 2), "ms"
            )

            total_time = max(s["end"] for s in segments)
            busy_time = sum(bursts)

            efficiency = (
                busy_time / total_time * 100
                if total_time > 0 else 0
            )

            print("Total Elapsed Time:", total_time, "ms")
            print("CPU Efficiency:", round(efficiency, 2), "%")

            print("\nGANTT CHART")

            draw_cpu_chart(segments)

        except Exception as error:

            print("Input Error:", error)


cpu_button.on_click(run_cpu)


# ============================================================
# CPU STEP-BY-STEP DISPLAY
# ============================================================

def render_cpu_step(change=None):

    with cpu_step_output:

        clear_output(wait=True)

        if not cpu_state["loaded"]:

            print("Click Load CPU Steps to begin.")
            return

        segments = cpu_state["segments"]
        current_time = cpu_play.value
        total = cpu_state["total"]

        running = "Completed"

        for seg in segments:

            if seg["start"] <= current_time < seg["end"]:

                running = seg["process"]
                break

        print("CPU SCHEDULING:", cpu_state["algorithm"])
        print("=" * 40)

        print("Current Time:", current_time, "ms")
        print("Total Time:", total, "ms")
        print("Currently Running:", running)

        if current_time == 0:

            print("Starting point. CPU execution has not started.")

        elif current_time == total:

            print("All processes have completed.")

        else:

            print("Showing execution up to time:", current_time)

        fig, ax = plt.subplots(figsize=(11, 3))

        for seg in segments:

            start = seg["start"]
            end = min(seg["end"], current_time)

            if end <= start:
                continue

            ax.barh(
                0,
                end - start,
                left=start,
                height=0.5,
                edgecolor="black",
                alpha=0.85
            )

            ax.text(
                (start + end) / 2,
                0,
                seg["process"],
                ha="center",
                va="center",
                fontsize=9
            )

        ax.axvline(
            current_time,
            linestyle="--",
            linewidth=2,
            label="Current Time"
        )

        ax.set_xlim(0, max(total, 1))
        ax.set_yticks([])
        ax.set_xlabel("CPU Time (ms)")
        ax.set_title("CPU Gantt Chart - Step by Step")
        ax.grid(axis="x", alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.show()
        plt.close(fig)


def load_cpu_steps(button):

    try:

        bursts = [
            int(x.strip())
            for x in cpu_burst.value.split(",")
            if x.strip()
        ]

        arrivals = [
            int(x.strip())
            for x in cpu_arrival.value.split(",")
            if x.strip()
        ]

        priorities = [
            int(x.strip())
            for x in cpu_priority.value.split(",")
            if x.strip()
        ]

        if not bursts or any(x <= 0 for x in bursts):
            raise ValueError("Burst times must be positive.")

        if len(arrivals) != len(bursts):
            raise ValueError("Enter one arrival time per process.")

        if len(priorities) != len(bursts):
            raise ValueError("Enter one priority per process.")

        if any(x < 0 for x in arrivals):
            raise ValueError("Arrival times cannot be negative.")

        segments, results = cpu_scheduling(
            bursts,
            arrivals,
            priorities,
            cpu_algorithm.value,
            cpu_quantum.value
        )

        total = max(seg["end"] for seg in segments)

        cpu_state["loaded"] = True
        cpu_state["segments"] = segments
        cpu_state["algorithm"] = cpu_algorithm.value
        cpu_state["total"] = total

        cpu_play.max = total
        cpu_play.value = 0

        render_cpu_step()

    except Exception as error:

        with cpu_step_output:
            clear_output(wait=True)
            print("Input Error:", error)


def cpu_prev_click(button):

    cpu_play.value = max(0, cpu_play.value - 1)


def cpu_next_click(button):

    cpu_play.value = min(
        cpu_play.max,
        cpu_play.value + 1
    )


cpu_load.on_click(load_cpu_steps)
cpu_previous.on_click(cpu_prev_click)
cpu_next.on_click(cpu_next_click)

cpu_play.observe(
    render_cpu_step,
    names="value"
)


# ============================================================
# FINAL COMBINED INTERFACE
# ============================================================

display(HTML("""
<div style="
    background:#1976D2;
    color:white;
    padding:22px;
    border-radius:12px;
    text-align:center;
">
    <h1>OS Scheduling Visualizer</h1>
    <p>Disk Scheduling and CPU Scheduling Algorithms</p>
    <p>Algorithm Execution and Step-by-Step Demonstration</p>
</div>
"""))


# DISK PANEL

display(HTML("""
<h2 style="color:#1976D2">1. Disk Scheduling</h2>
<p>Choose an algorithm and run the disk scheduling simulation.</p>
"""))

disk_panel = widgets.VBox([

    disk_algorithm,
    disk_queue,
    disk_head,
    disk_size,
    disk_direction,

    disk_button,
    disk_output,

    widgets.HTML("<hr><b>Step-by-Step Disk Demonstration</b>"),

    disk_load,

    widgets.HBox([
        disk_previous,
        disk_play,
        disk_next
    ]),

    disk_step_output

], layout=widgets.Layout(
    border="1px solid #cccccc",
    padding="15px",
    margin="5px 0 20px 0"
))

display(disk_panel)


# CPU PANEL

display(HTML("""
<h2 style="color:#1976D2">2. CPU Scheduling</h2>
<p>Choose an algorithm and run the CPU scheduling simulation.</p>
"""))

cpu_panel = widgets.VBox([

    cpu_algorithm,
    cpu_burst,
    cpu_arrival,
    cpu_priority,
    cpu_quantum,

    cpu_button,
    cpu_output,

    widgets.HTML("<hr><b>Step-by-Step CPU Demonstration</b>"),

    cpu_load,

    widgets.HBox([
        cpu_previous,
        cpu_play,
        cpu_next
    ]),

    cpu_step_output

], layout=widgets.Layout(
    border="1px solid #cccccc",
    padding="15px",
    margin="5px 0 20px 0"
))

display(cpu_panel)


display(HTML("""
<div style="
    background:#eeeeee;
    padding:15px;
    border-radius:8px;
    text-align:center;
">
    <b>OS Scheduling Visualizer</b>
    <p>Project Ready</p>
</div>
"""))

# CONNECT RUN BUTTONS
disk_button.on_click(run_disk)
cpu_button.on_click(run_cpu)

print("OS Scheduling Visualizer is ready!")


OS Scheduling Visualizer is ready!
